In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T

In [3]:
spark = SparkSession.builder.appName(
    "RDD and UDFs"
).getOrCreate()

spark.sparkContext

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/usr/local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/14 21:05:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkContext master=local[*] appName=RDD and UDFs>

In [4]:
sc = spark.sparkContext

# **RDD (Resilient Distributed Dataset)**

![](../images/RDD.png)

An **RDD (Resilient Distributed Dataset)** is the fundamental building block and core abstraction of Apache Spark. Introduced in the original 2012 Spark paper, it represents an immutable, fault-tolerant, distributed collection of objects that can be processed in parallel across a cluster.

---

### **🏛️ The Three Pillars of an RDD**

1. **Resilient (Fault-Tolerant):**
* If a partition of data on a worker node is lost due to a node failure, Spark doesn't need to replicate data across the network (like Hadoop).
* Instead, it reconstructs the lost partition by replaying the **Lineage Graph** (the deterministic sequence of transformations that created it).


2. **Distributed:**
* Data is automatically partitioned and spread across multiple nodes in the cluster, allowing parallel processing.


3. **Dataset:**
* A collection of partitioned data elements (which can be primitive types, tuples, or custom Python/Java/Scala objects).

---

### **⚙️ Core Characteristics & Execution Model**

#### **1. Immutability**

Once created, an RDD cannot be changed. Every transformation applied to an RDD (like `.map()` or `.filter()`) produces a **brand-new RDD**, preserving lineage and making fault-tolerance trivial.

#### **2. Lazy Evaluation**

Transformations on RDDs do not compute results immediately. Spark builds a **Directed Acyclic Graph (DAG)** of transformations and only triggers physical computation when an **Action** (e.g., `.collect()`, `.count()`, `.saveAsTextFile()`) is called.

* **Transformations (Lazy):** `map`, `filter`, `flatMap`, `distinct` (returns a new RDD).
* **Actions (Eager):** `collect`, `take`, `reduce`, `count`, `save` (triggers cluster execution).

---

### **🔄 RDDs vs. DataFrames & Datasets**

While modern PySpark code almost exclusively uses **DataFrames** and **Datasets**, RDDs still operate silently underneath every DataFrame.

| Feature | **RDD** | **DataFrame / Dataset** |
| --- | --- | --- |
| **API Style** | Low-level functional programming (lambdas, closures). | High-level relational API (`select`, `groupby`, SQL expressions). |
| **Optimization** | **None.** Spark executes your code exactly as written (black-box closures). | **Catalyst Optimizer & Tungsten.** Automatically optimizes query plans, predicate pushdown, and binary memory layouts. |
| **Type Safety** | Compile-time/runtime object checking depending on language. | Schema-enforced tabular structure. |
| **When to use** | Unstructured binary data, custom low-level partitioning control, or legacy codebases. | 99% of modern big data ETL, analytics, and machine learning pipelines. |

### **💡 Example: Low-Level RDD Usage in PySpark**

Even in a DataFrame workflow, you can drop down to the underlying RDD layer using `.rdd`:

```python
# Convert a DataFrame to an RDD of Row objects
rdd = full_log.rdd

# Apply low-level Python transformations using a lambda function
filtered_rdd = rdd.filter(lambda row: row["duration_seconds"] > 3600)

# Trigger an action
print(filtered_rdd.count())
```

## **Map Operation**

In [5]:
collection = [1, "two", 3.0, ("four", 4), {"five": 5}]

In [7]:
collection_rdd = sc.parallelize(collection)

collection_rdd

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:326

In [8]:
from py4j.protocol import Py4JJavaError

In [9]:
def add_one(value):
    return value + 1

In [11]:
collection_rdd = collection_rdd.map(add_one)

try:
    print(collection_rdd.collect())
except Py4JJavaError:
    pass

26/11/14 17:16:01 ERROR Executor: Exception in task 1.0 in stage 0.0 (TID 1)/ 4]
org.apache.spark.api.python.PythonException: [PYTHON_EXCEPTION] An exception was thrown from the Python worker: Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 3622, in main
    process()
  File "/usr/local/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 3612, in process
    serializer.dump_stream(out_iter, outfile)
  File "/usr/local/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/serializers.py", line 257, in dump_stream
    vs = list(itertools.islice(iterator, batch))
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/util.py", line 159, in wrapper
    return f(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1717/3587218469.py", line 2, in add_one
TypeE

PythonException: An exception was thrown from the Python worker:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 3622, in main
    process()
  File "/usr/local/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 3612, in process
    serializer.dump_stream(out_iter, outfile)
  File "/usr/local/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/serializers.py", line 257, in dump_stream
    vs = list(itertools.islice(iterator, batch))
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/util.py", line 159, in wrapper
    return f(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1717/3587218469.py", line 2, in add_one
TypeError: can only concatenate tuple (not "int") to tuple

![](../images/RDD_map.png)

In [12]:
collection_rdd = sc.parallelize(collection)

def safe_add_one(value):
    try:
        return value + 1
    except TypeError:
        return value

In [14]:
collection_rdd = collection_rdd.map(safe_add_one)

try:
    print(collection_rdd.collect())
except Py4JJavaError:
    pass

[2, 'two', 4.0, ('four', 4), {'five': 5}]


## **Filter Operation**

In [15]:
collection_rdd = sc.parallelize(collection)

collection_rdd = collection_rdd.filter(
    lambda elm: isinstance(elm, (float, int))
)

print(collection_rdd.collect())

[1, 3.0]


## **Reduce Operation**

In [16]:
from operator import add

In [17]:
collection_rdd = sc.parallelize([4, 7, 9, 1, 3])

print(collection_rdd.reduce(add))

24


## **DataFrame is an RDD**

In [18]:
df = spark.createDataFrame([[1], [2], [3]], schema=["column"])

df.show()

+------+
|column|
+------+
|     1|
|     2|
|     3|
+------+



In [19]:
df.rdd

MapPartitionsRDD[19] at javaToPython at DirectMethodHandleAccessor.java:103

In [20]:
df.rdd.collect()

[Row(column=1), Row(column=2), Row(column=3)]

# **UDFs (User-Defined Function)**

In Apache Spark, a **UDF (User-Defined Function)** allows you to write custom transformation logic in Python (or Scala/Java) and apply it row-by-row across a distributed DataFrame.

While UDFs provide immense flexibility when built-in SQL functions aren't enough, they are also **one of the most notorious performance bottlenecks** in PySpark if used incorrectly.

---

## **⚙️ How Standard PySpark UDFs Work (And Why They're Slow)**

Spark is written in Scala/Java and runs on the JVM (Java Virtual Machine), leveraging Project Tungsten for lightning-fast, off-heap memory processing.

When you use a standard Python UDF:

1. **Serialization Tax:** Spark must serialize rows from the JVM into Python objects.
2. **Context Switching:** Data is sent across a local socket connection (via Py4J) to a separate Python worker process.
3. **Row-by-Row Execution:** Python evaluates the function on each row individually (interpreting bytecode, incurring massive Python interpreter overhead).
4. **Deserialization Back:** The result is serialized back into JVM format.

Furthermore, standard UDFs act as a **black box** to the Catalyst Optimizer. Spark cannot optimize, reorder, or push down filters through a standard Python UDF because it has no visibility into what the Python code actually does.

---

## **💻 1. Standard PySpark UDF Example**

Here is how you define and register a standard Python UDF using `pyspark.sql.functions.udf`:

```python
import pyspark.sql.functions as F
from pyspark.sql.types import StringType


# 1. Define your custom Python function
def categorize_capacity(capacity_gb):
    if capacity_gb is None:
        return "Unknown"
    elif capacity_gb < 2000:
        return "Small"
    elif capacity_gb < 8000:
        return "Medium"
    else:
        print(f"Large Drive Detected: {capacity_gb} GB")
        return "Large"


# 2. Register the UDF with an explicit return type (crucial for Spark's Catalyst compiler)
categorize_udf = F.udf(categorize_capacity, StringType())

# 3. Apply the UDF in a DataFrame transformation
df_categorized = full_data.select(
    "model", "capacity_GB", categorize_udf("capacity_GB").alias("size_category")
)

```

---

## **🚀 2. The High-Performance Alternative: Pandas UDFs (Vectorized UDFs)**

To solve the row-by-row overhead of standard UDFs, Spark introduced **Pandas UDFs** (powered by **Apache Arrow**).

Instead of passing data row-by-row, Pandas UDFs transfer data in vectorized batches using Apache Arrow's in-memory columnar format. This allows Python to execute operations using vectorized NumPy/Pandas code, achieving speedups of **10x to 100x**.

### **Example: Pandas UDF**

```python
from pyspark.sql.functions import pandas_udf
import pandas as pd


# Define a Pandas UDF taking a Pandas Series and returning a Pandas Series
@pandas_udf("string")
def categorize_pandas_udf(capacity_series: pd.Series) -> pd.Series:
    # Vectorized operations using pandas/numpy logic (no row-by-row python loops)
    return pd.select(
        [
            capacity_series < 2000,
            (capacity_series >= 2000) & (capacity_series < 8000),
            capacity_series >= 8000,
        ],
        ["Small", "Medium", "Large"],
        default="Unknown",
    )


# Apply it just like a regular column expression
df_vectorized = full_data.select(
    "model",
    "capacity_GB",
    categorize_pandas_udf("capacity_GB").alias("size_category"),
)

```

---

## **📋 Best Practices & Engineering Guidelines for UDFs**

1. **The #1 Rule of UDFs:** *Don't use them if a built-in Spark SQL function exists.*
* Built-in functions (`F.col`, `F.when`, `F.regexp_extract`, `F.coalesce`, mathematical expressions) are written in optimized Java/Scala, execute directly on Tungsten binary memory, and allow full Catalyst optimization.


2. **Always Declare Return Types:** In standard UDFs, always specify the return type (`StringType()`, `LongType()`, etc.). Letting Spark guess can lead to runtime crashes or type mismatch errors.
3. **Use Pandas UDFs for Heavy Python Libraries:** If you need to run machine learning model inference (e.g., PyTorch, Scikit-Learn), cryptographic hashing, or complex statistical routines via Pandas/NumPy, always use `@pandas_udf` to leverage Apache Arrow vectorization.
4. **Prefer SQL Expressions (`selectExpr`):** For simple arithmetic, conditional logic (`CASE WHEN`), or string manipulations, `selectExpr()` or standard expressions are cleaner, faster, and easier for the optimizer to handle than Python UDFs.

In [31]:
fractions = [[x, y] for x in range(100) for y in range(1, 100)]

frac_df = spark.createDataFrame(
    fractions, ["numerator", "denominator"]
)

In [32]:
frac_df = frac_df.select(
    F.array(F.col('numerator'), F.col('denominator')).alias('fraction')
)

frac_df.show(5)

+--------+
|fraction|
+--------+
|  [0, 1]|
|  [0, 2]|
|  [0, 3]|
|  [0, 4]|
|  [0, 5]|
+--------+
only showing top 5 rows


In [24]:
from fractions import Fraction
from typing import Tuple, Optional

In [25]:
Frac = Tuple[int, int]

In [26]:
def py_reduce_fraction(frac: Frac) -> Optional[Frac]:
    """Reduce a fraction represented as a 2-tuple of integers."""

    num, denom = frac
    if denom:
        answer = Fraction(num, denom)
        return answer.numerator, answer.denominator
    return None

In [27]:
assert py_reduce_fraction((3, 6)) == (1, 2)
assert py_reduce_fraction((1, 0)) is None

In [28]:
def py_fraction_to_float(frac: Frac) -> Optional[float]:
    """Transforms a fraction represented as a 2-tuple of integers into a float."""

    num, denom = frac
    if denom:
        return num / denom
    return None

In [29]:
assert py_fraction_to_float((2, 8)) == 0.25
assert py_fraction_to_float((10, 0)) is None

In [33]:
sparkFrac = T.ArrayType(T.LongType())

reduce_fraction = F.udf(py_reduce_fraction, sparkFrac)

frac_df = frac_df.withColumn(
    'reduced_fraction', reduce_fraction(F.col('fraction'))
)

frac_df.show(5, False)

+--------+----------------+
|fraction|reduced_fraction|
+--------+----------------+
|[0, 1]  |[0, 1]          |
|[0, 2]  |[0, 1]          |
|[0, 3]  |[0, 1]          |
|[0, 4]  |[0, 1]          |
|[0, 5]  |[0, 1]          |
+--------+----------------+
only showing top 5 rows


In [34]:
@F.udf(T.DoubleType())
def fraction_to_float(frac: Frac) -> Optional[float]:
    """Transforms a fraction represented as a 2-tuple of integers into a float."""

    num, denom = frac
    if denom:
        return num / denom
    
    return None

In [35]:
frac_df = frac_df.withColumn(
    "fraction_float", fraction_to_float(F.col("reduced_fraction"))
)

frac_df.select("reduced_fraction", "fraction_float").distinct().show(5, False)

[Stage 12:>                                                         (0 + 4) / 4]

+----------------+-------------------+
|reduced_fraction|fraction_float     |
+----------------+-------------------+
|[3, 50]         |0.06               |
|[3, 67]         |0.04477611940298507|
|[7, 76]         |0.09210526315789473|
|[9, 23]         |0.391304347826087  |
|[9, 25]         |0.36               |
+----------------+-------------------+
only showing top 5 rows


# **pandas UDFs**

A **Pandas UDF** (also known as a **Vectorized UDF**) is a high-performance user-defined function in PySpark that leverages **Apache Arrow** to transfer data between the JVM and Python worker processes, eliminating the massive serialization overhead of standard row-by-row UDFs.

---

### **🚀 The Secret Sauce: Apache Arrow**

In standard PySpark UDFs, Spark serializes Java/Scala rows into Python objects one by one. For millions of rows, this creates a severe CPU bottleneck.

**Pandas UDFs solve this using Apache Arrow:**

* **In-Memory Columnar Format:** Data is transferred in columnar batches rather than individual rows.
* **Zero-Copy Interoperability:** Arrow memory can be shared between JVM and Python processes with almost zero serialization/deserialization cost.
* **Vectorized Execution:** Python code operates on entire Pandas `Series` or `DataFrame` chunks at once, allowing NumPy and Pandas C-optimized backends to do the heavy lifting.

---

### **📊 Types of Pandas UDFs (Spark 3.x+)**

Depending on your use case, Pandas UDFs come in a few architectural flavors:

1. **Scalar (Series to Series):** Takes one or more Pandas `Series` and returns a Pandas `Series` of the same length. Used for column-wise transformations.
2. **Iterator of Series to Iterator of Series:** Similar to scalar, but processes data as an *iterator* of batches. Ideal if your function needs to initialize a heavy resource (like loading a machine learning model into memory) once per partition rather than once per batch.
3. **Grouped Map (Pandas Function APIs):** Takes a grouped DataFrame (`df.groupBy().applyInPandas(...)`), passes each group as a Pandas `DataFrame`, and returns a Pandas `DataFrame`. Essential for distributed machine learning training or running group-specific calculations.

---

### **💻 Code Example: Scalar Pandas UDF**

Here is how you implement a scalar Pandas UDF to compute a custom normalization metric across a column:

```python
import pyspark.sql.functions as F
from pyspark.sql.functions import pandas_udf
import pandas as pd


# 1. Define the Pandas UDF using the @pandas_udf decorator
# Input: Pandas Series (float64) -> Output: Pandas Series (float64)
@pandas_udf("double")
def compute_z_score(values: pd.Series) -> pd.Series:
    # Vectorized calculation using NumPy / Pandas (no python loops)
    mean = values.mean()
    std = values.std()

    if std == 0 or pd.isna(std):
        return values * 0.0  # Handle zero variance edge case

    return (values - mean) / std


# 2. Ensure Apache Arrow optimization is enabled (usually default in modern Spark)
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

# 3. Apply the Pandas UDF just like a built-in function
df_normalized = full_data.select(
    "model",
    "capacity_GB",
    compute_z_score("capacity_GB").alias("capacity_z_score"),
)

df_normalized.show(5, truncate=False)

```

---

### **🧠 Advanced Pattern: Iterator of Series (Resource Pooling)**

If your Pandas UDF performs a heavy operation—such as loading a scikit-learn model or connecting to an external resource—instantiating it on every batch inside a partition kills performance. Using an iterator allows you to initialize the resource **once per task/partition**:

```python
from typing import Iterator


@pandas_udf("double")
def predict_with_loaded_model(
    iterator: Iterator[pd.Series],
) -> Iterator[pd.Series]:
    # Load model ONCE per task/partition worker
    # model = pickle.load(open("model.pkl", "rb"))

    for batch in iterator:
        # Perform vector prediction on each batch
        # result = model.predict(batch)
        result = batch * 1.05  # Placeholder vectorized operation
        yield pd.Series(result)


# Applied across partitions
df_predicted = full_data.select(
    predict_with_loaded_model("capacity_GB").alias("prediction")
)

```

---

### **📋 Best Practices & Gotchas**

1. **Type Hints are Mandatory:** Spark requires type hints (`pd.Series -> pd.Series`) in your Python function signature to validate schemas at analysis time.
2. **Arrow Compatibility Constraints:** Not all SQL types are natively supported by Arrow (e.g., complex nested maps with non-string keys or certain calendar intervals can cause fallback warnings). Stick to primitive types (`long`, `double`, `string`, `boolean`, `date`) for peak performance.
3. **Memory Management on Workers:** Because Pandas UDFs load entire columnar batches into Python's memory space on the executor, watch out for `OutOfMemory (OOM)` errors if your batches or groups are massive. Adjust `spark.sql.execution.arrow.maxRecordsPerBatch` if necessary to control batch sizes.

In [6]:
df_gsod = (
    T.reduce(
        lambda x, y: x.unionByName(y, allowMissingColumns=True),
        [
            spark.read.parquet(f"../data/gsod_noaa/gsod{year}.parquet")
            for year in range(2010, 2021)
        ]
    )
    .dropna(subset=["year", "mo", "da", "temp"])
    .where(F.col("temp") != 9999.9)
    .drop("date")
)

![](../images/Python_UDF_to_a_pandas_scalar_UDF.png)

In [10]:
import pandas as pd

In [12]:
@F.pandas_udf(T.DoubleType())
def f_to_c(degrees: pd.Series) -> pd.Series:
    """Transforms Farhenheit to Celcius."""

    return (degrees - 32) * 5/ 9

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/functions.py:777: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/typehints.py:60: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [14]:
df_gsod = df_gsod.withColumn(
    "temp_c", f_to_c(F.col("temp"))
)

df_gsod.select(
    "temp",
    "temp_c"
).distinct().show(5)

[Stage 15:======================================================> (43 + 1) / 44]

+----+-------------------+
|temp|             temp_c|
+----+-------------------+
|29.6|-1.3333333333333326|
|37.2| 2.8888888888888906|
|53.5| 11.944444444444445|
|70.4| 21.333333333333336|
|24.7| -4.055555555555555|
+----+-------------------+
only showing top 5 rows


In [15]:
from time import sleep
from typing import Iterator

In [22]:
@F.pandas_udf(T.DoubleType())
def f_to_c2(degrees: Iterator[pd.Series]) -> Iterator[pd.Series]:
    """Transforms Farhenheit to Celcius."""
    sleep(5)
    for batch in degrees:
        yield (batch - 32) * 5 / 9

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/functions.py:777: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/typehints.py:60: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [23]:
df_gsod.select(
    "temp", f_to_c2(F.col("temp")).alias("temp_c2")
).distinct().show(5)

[Stage 20:=====================================================>  (42 + 2) / 44]

+----+-------------------+
|temp|            temp_c2|
+----+-------------------+
|29.6|-1.3333333333333326|
|37.2| 2.8888888888888906|
|53.5| 11.944444444444445|
|70.4| 21.333333333333336|
|24.7| -4.055555555555555|
+----+-------------------+
only showing top 5 rows


![](../images/transformation_of_three_Series.png)

In [24]:
from typing import Tuple

In [26]:
@F.pandas_udf(T.DateType())
def create_date(
    year_mo_da: Iterator[Tuple[pd.Series, pd.Series, pd.Series]]
) -> Iterator[pd.Series]:
    """Merges three cols (representing Y-M-D of a date) into a Date col."""
    for year, mo, da in year_mo_da:
        yield pd.to_datetime(
            pd.DataFrame(dict(year=year, month=mo, day=da))
        )

In [27]:
df_gsod.select(
    "year", "mo", "da",
    create_date(F.col("year"), F.col("mo"), F.col("da")).alias("date")
).distinct().show(5)

[Stage 23:======================================================> (43 + 1) / 44]

+----+---+---+----------+
|year| mo| da|      date|
+----+---+---+----------+
|2010| 08| 31|2010-08-31|
|2010| 12| 20|2010-12-20|
|2010| 12| 16|2010-12-16|
|2010| 04| 03|2010-04-03|
|2010| 03| 17|2010-03-17|
+----+---+---+----------+
only showing top 5 rows


> **TIP :** By default, Spark will aim for 10,000 records per batch. You can customize the maximum size of each batch using the spark.sql.execution.arrow.maxRecordsPerBatch config when creating the SparkSession object; 10,000 records is a good balance for most jobs. If you are working with memory-starved executors, you might want to reduce this.

The **Split-Apply-Combine** pattern—popularized in data analysis by Hadley Wickham and foundational to tools like Pandas—is just as critical in distributed computing. In Apache Spark, this pattern governs how you group data, perform independent operations on those groups, and recombine them into a unified dataset.

Depending on your analytical goal and performance constraints, PySpark implements this pattern across three distinct architectural layers:

---

### **1. Native Relational Aggregations (`groupBy().agg()`)**

* **Best for:** Standard summary statistics (sums, averages, counts, distinct counts).
* **Execution:** Fully optimized by the Catalyst Optimizer. Spark pushes down aggregations to map stages before shuffling data across the network to the correct reducer nodes.

```python
import pyspark.sql.functions as F

# SPLIT: Group by hard drive model
# APPLY: Compute total drive days and average capacity
# COMBINE: Return aggregated table
summary_df = (
    full_data.groupBy("model")
    .agg(
        F.count("*").alias("total_days"),
        F.avg("capacity_GB").alias("avg_capacity"),
        F.sum("failure").alias("total_failures"),
    )
    .orderBy(F.desc("total_failures"))
)

```

---

### **2. Grouped Window Functions (`Window.partitionBy()`)**

* **Best for:** Row-level transformations that require group context (e.g., ranking transactions within an state, calculating running totals, or finding moving averages) **without collapsing your DataFrame**.
* **Execution:** Avoids full aggregation collapse. Data is shuffled by partition keys so rows sharing a group key land on the same executor to compute sequential ranks or offsets.

```python
from pyspark.sql.window import Window

# Define the split boundary (Partition by model, order by date)
window_spec = Window.partitionBy("model").orderBy("date")

# Apply window transformations across the split groups and combine back into row stream
trend_df = full_data.withColumn(
    # Cumulative sum of failures per model over time
    "cumulative_failures",
    F.sum("failure").over(window_spec),
).withColumn(
    # Rank days of operation per model
    "operational_day_rank",
    F.row_number().over(window_spec),
)

```

---

### **3. Grouped Map Pandas UDFs (`applyInPandas`)**

* **Best for:** Complex operations requiring custom Python/NumPy/SciPy/Scikit-Learn logic per group (e.g., fitting a linear regression model per hard drive model, calculating custom covariance matrices, or running specialized time-series forecasting).
* **Execution:** Spark splits data by key, serializes each group into a Pandas DataFrame using Apache Arrow, passes it to a Python worker where your custom function executes, and combines the resulting Pandas DataFrames back into a Spark DataFrame.

```python
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import DoubleType, StringType, StructField, StructType
import pandas as pd

# 1. Define the explicit output schema for the combined DataFrame
output_schema = StructType(
    [
        StructField("model", StringType(), True),
        StructField("capacity_mean", DoubleType(), True),
        StructField("failure_rate_std", DoubleType(), True),
    ]
)


# 2. Define the custom Pandas function operating on an individual group's DataFrame
@pandas_udf(output_schema)
def compute_group_metrics(pdf: pd.DataFrame) -> pd.DataFrame:
    # 'pdf' represents all rows for a single hard drive model as a Pandas DataFrame
    model_name = pdf["model"].iloc[0]
    mean_cap = pdf["capacity_GB"].mean()
    std_failure = pdf["failure"].std()

    return pd.DataFrame(
        {
            "model": [model_name],
            "capacity_mean": [mean_cap],
            "failure_rate_std": [std_failure],
        }
    )


# 3. SPLIT, APPLY, and COMBINE using applyInPandas
grouped_metrics_df = full_data.groupBy("model").applyInPandas(
    compute_group_metrics, schema=output_schema
)

```

---

### **⚠️ Performance & Architectural Trade-offs**

1. **The Shuffle Cost:** All three patterns require a **wide transformation (shuffle)** across the cluster to route rows sharing the same group key to the same executor node. If your data is heavily skewed (e.g., 90% of your logs belong to a single popular drive model), one executor will bottleneck the entire stage while others sit idle.
2. **Prefer Native over Pandas UDFs:** If your logic can be written using native Spark SQL functions (`F.sum`, `F.avg`, `F.expr`), always choose Native. It avoids Arrow serialization overhead and keeps execution entirely inside Project Tungsten's optimized memory engine.
3. **Memory Limits in `applyInPandas`:** When using `applyInPandas`, ensure that no single group exceeds the memory capacity of a single Python worker container. If a single group (e.g., all logs for one massive manufacturer) cannot fit into executor memory, Spark will throw an Out-Of-Memory (OOM) error.

![](../images/Split-apply-combine.png)

## **Group aggregate UDFs**

In [28]:
from sklearn.linear_model import LinearRegression

In [33]:
@F.pandas_udf(T.DoubleType())
def rate_of_change_temp(day: pd.Series, temp: pd.Series) -> float:
    """Returns the slope of the daily temperature for a given period of time."""

    return (
        LinearRegression()
        .fit(X=day.astype(int).values.reshape(-1, 1), y=temp)
        .coef_[0]
    )

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/functions.py:777: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/typehints.py:60: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [34]:
result = df_gsod.groupby("stn", "year", "mo").agg(
    rate_of_change_temp(df_gsod['da'], df_gsod['temp']).alias('rt_chg_temp')
)

In [35]:
result.show(5, False)

[Stage 31:>                                                         (0 + 1) / 1]

+------+----+---+-------------------+
|stn   |year|mo |rt_chg_temp        |
+------+----+---+-------------------+
|007032|2013|01 |-0.2795031055900618|
|008268|2010|07 |-2.1999999999999877|
|008400|2010|02 |1.7828571428571436 |
|008400|2010|08 |0.8514508067970873 |
|008401|2011|03 |0.7514516129032257 |
+------+----+---+-------------------+
only showing top 5 rows


Traceback (most recent call last):                                              
  File "/usr/local/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 262, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 95, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


## **Group map UDF**

In [37]:
def scale_temperature(temp_by_day: pd.DataFrame) -> pd.DataFrame:
    """
    Returns a simple normalization of the temperature for a site.
    If the temperature is constant for the whole window, defaults to 0.5.
    """

    temp = temp_by_day.temp
    answer = temp_by_day[["stn", "year", "mo", "da", "temp"]]
    if temp.min() == temp.max():
        return answer.assign(temp_norm=0.5)
    return answer.assign(
        temp_norm=(temp - temp.min()) / (temp.max() - temp.min())
    )

In [38]:
gsod_map = df_gsod.groupby("stn", "year", "mo").applyInPandas(
    scale_temperature,
    schema=(
        "stn string, year string, mo string, "
        "da string, temp double, temp_norm double"
    ),
)

gsod_map.show(5, False)

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/typehints.py:505: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/functions.py:777: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
[Stage 32:======================================================> (43 + 1) / 44]

+------+----+---+---+----+------------------+
|stn   |year|mo |da |temp|temp_norm         |
+------+----+---+---+----+------------------+
|008268|2010|07 |21 |89.6|1.0               |
|008268|2010|07 |22 |87.4|0.0               |
|008401|2011|11 |24 |61.3|0.52              |
|008401|2011|11 |05 |66.0|0.7080000000000001|
|008401|2011|11 |01 |68.2|0.7960000000000003|
+------+----+---+---+----+------------------+
only showing top 5 rows


Traceback (most recent call last):                                              
  File "/usr/local/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 262, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 95, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


![](../images/summary_UDF.png)

# **Window functions**

In [39]:
df_gsod = spark.read.parquet('../data/gsod_parquet')

In [40]:
coldest_temp = df_gsod.groupby("year").agg(
    F.min("temp").alias("temp")
)

coldest_temp.orderBy("temp").show()

+----+------+
|year|  temp|
+----+------+
|2019|-114.7|
|2017|-114.7|
|2018|-113.5|
+----+------+



In [41]:
coldest_when = df_gsod.join(
    coldest_temp, how='left_semi', on=['year', 'temp']
).select("stn", "year", "mo", "da", "temp")

coldest_when.orderBy("year", "mo", "da").show()

+------+----+---+---+------+
|   stn|year| mo| da|  temp|
+------+----+---+---+------+
|896250|2017| 06| 20|-114.7|
|896060|2018| 08| 27|-113.5|
|895770|2019| 06| 15|-114.7|
+------+----+---+---+------+



![](../images/self-join.png)

In [42]:
from pyspark.sql.window import Window

In [43]:
each_year = Window.partitionBy('year')

In [44]:
each_year

In [45]:
(
    df_gsod
    .withColumn('min_temp', F.min('temp').over(each_year))
    .where('temp = min_temp')
    .select("year", "mo", "da", "stn", "temp")
    .orderBy("year", "mo", "da")
    .show()
)

[Stage 47:>                                                         (0 + 3) / 3]

+----+---+---+------+------+
|year| mo| da|   stn|  temp|
+----+---+---+------+------+
|2017| 06| 20|896250|-114.7|
|2018| 08| 27|896060|-113.5|
|2019| 06| 15|895770|-114.7|
+----+---+---+------+------+



![](../images/windows_partition.png)

## **Ranking functions**

In [46]:
temp_per_month_asc = Window.partitionBy("mo").orderBy("count_temp")

In [59]:
df_gsod.withColumn(
    "rank_temp", F.rank().over(temp_per_month_asc)
).select("stn", "year", "mo", "da", "temp", "count_temp", "rank_temp").show()

[Stage 81:>                                                         (0 + 1) / 1]

+------+----+---+---+-----+----------+---------+
|   stn|year| mo| da| temp|count_temp|rank_temp|
+------+----+---+---+-----+----------+---------+
|644600|2018| 01| 11| 78.3|         4|        1|
|997344|2017| 01| 17| 43.9|         4|        1|
|141180|2018| 01| 15| 29.1|         4|        1|
|617690|2017| 01| 23| 82.9|         4|        1|
|141180|2018| 01| 25| 36.0|         4|        1|
|896250|2017| 01| 02|-22.9|         4|        1|
|013760|2018| 01| 28| 21.6|         4|        1|
|420830|2017| 01| 24| 53.3|         4|        1|
|013760|2018| 01| 01| 24.3|         4|        1|
|647580|2017| 01| 14| 83.3|         4|        1|
|141180|2018| 01| 29| 36.5|         4|        1|
|915710|2017| 01| 30| 77.6|         4|        1|
|896250|2018| 01| 23|-24.5|         4|        1|
|014890|2017| 01| 04| 22.0|         4|        1|
|480450|2018| 01| 11| 59.2|         4|        1|
|990058|2017| 01| 17| 76.0|         4|        1|
|412490|2018| 01| 16| 64.2|         4|        1|
|141180|2017| 01| 26

In [60]:
df_gsod.withColumn(
    "rank_temp", F.dense_rank().over(temp_per_month_asc)
).select("stn", "year", "mo", "da", "temp", "count_temp", "rank_temp").show()

[Stage 84:>                                                         (0 + 1) / 1]

+------+----+---+---+-----+----------+---------+
|   stn|year| mo| da| temp|count_temp|rank_temp|
+------+----+---+---+-----+----------+---------+
|644600|2018| 01| 11| 78.3|         4|        1|
|997344|2017| 01| 17| 43.9|         4|        1|
|141180|2018| 01| 15| 29.1|         4|        1|
|617690|2017| 01| 23| 82.9|         4|        1|
|141180|2018| 01| 25| 36.0|         4|        1|
|896250|2017| 01| 02|-22.9|         4|        1|
|013760|2018| 01| 28| 21.6|         4|        1|
|420830|2017| 01| 24| 53.3|         4|        1|
|013760|2018| 01| 01| 24.3|         4|        1|
|647580|2017| 01| 14| 83.3|         4|        1|
|141180|2018| 01| 29| 36.5|         4|        1|
|915710|2017| 01| 30| 77.6|         4|        1|
|896250|2018| 01| 23|-24.5|         4|        1|
|014890|2017| 01| 04| 22.0|         4|        1|
|480450|2018| 01| 11| 59.2|         4|        1|
|990058|2017| 01| 17| 76.0|         4|        1|
|412490|2018| 01| 16| 64.2|         4|        1|
|141180|2017| 01| 26

In [61]:
temp_each_year = each_year.orderBy("temp")

In [62]:
df_gsod.withColumn(
    "rank_temp", F.percent_rank().over(temp_each_year)
).select("stn", "year", "mo", "da", "temp", "count_temp", "rank_temp").show()

[Stage 87:>                                                         (0 + 1) / 1]

+------+----+---+---+------+----------+--------------------+
|   stn|year| mo| da|  temp|count_temp|           rank_temp|
+------+----+---+---+------+----------+--------------------+
|895770|2019| 06| 15|-114.7|        21|                 0.0|
|895770|2019| 06| 14|-113.6|        20|2.406129084494351E-7|
|896060|2019| 06| 07|-112.4|         4|4.812258168988702E-7|
|896060|2019| 06| 08|-111.3|         4|7.218387253483052E-7|
|895770|2019| 06| 17|-110.7|        20|9.624516337977404E-7|
|895770|2019| 05| 25|-110.3|        20|1.203064542247175...|
|896060|2019| 06| 09|-109.0|         4|1.443677450696610...|
|895770|2019| 06| 02|-108.6|        21|1.684290359146045...|
|895770|2019| 06| 18|-108.5|        20|1.924903267595480...|
|897340|2019| 07| 19|-108.5|         5|1.924903267595480...|
|895770|2019| 06| 05|-108.1|        20|2.406129084494350...|
|896060|2019| 06| 18|-107.9|         4|2.646741992943786E-6|
|896060|2019| 06| 06|-107.8|         4|2.887354901393221E-6|
|897340|2019| 07| 20|-10

In [63]:
df_gsod.withColumn(
    "rank_temp", F.ntile(2).over(temp_each_year)
).select("stn", "year", "mo", "da", "temp", "count_temp", "rank_temp").show()

[Stage 90:>                                                         (0 + 1) / 1]

+------+----+---+---+------+----------+---------+
|   stn|year| mo| da|  temp|count_temp|rank_temp|
+------+----+---+---+------+----------+---------+
|895770|2019| 06| 15|-114.7|        21|        1|
|895770|2019| 06| 14|-113.6|        20|        1|
|896060|2019| 06| 07|-112.4|         4|        1|
|896060|2019| 06| 08|-111.3|         4|        1|
|895770|2019| 06| 17|-110.7|        20|        1|
|895770|2019| 05| 25|-110.3|        20|        1|
|896060|2019| 06| 09|-109.0|         4|        1|
|895770|2019| 06| 02|-108.6|        21|        1|
|895770|2019| 06| 18|-108.5|        20|        1|
|897340|2019| 07| 19|-108.5|         5|        1|
|895770|2019| 06| 05|-108.1|        20|        1|
|896060|2019| 06| 18|-107.9|         4|        1|
|896060|2019| 06| 06|-107.8|         4|        1|
|897340|2019| 07| 20|-107.6|         7|        1|
|895770|2019| 06| 03|-107.6|        16|        1|
|896060|2019| 05| 21|-107.1|         4|        1|
|895770|2019| 05| 26|-106.8|        19|        1|


In [64]:
df_gsod.withColumn(
    "rank_temp", F.row_number().over(temp_each_year)
).select("stn", "year", "mo", "da", "temp", "count_temp", "rank_temp").show()

[Stage 93:>                                                         (0 + 1) / 1]

+------+----+---+---+------+----------+---------+
|   stn|year| mo| da|  temp|count_temp|rank_temp|
+------+----+---+---+------+----------+---------+
|895770|2019| 06| 15|-114.7|        21|        1|
|895770|2019| 06| 14|-113.6|        20|        2|
|896060|2019| 06| 07|-112.4|         4|        3|
|896060|2019| 06| 08|-111.3|         4|        4|
|895770|2019| 06| 17|-110.7|        20|        5|
|895770|2019| 05| 25|-110.3|        20|        6|
|896060|2019| 06| 09|-109.0|         4|        7|
|895770|2019| 06| 02|-108.6|        21|        8|
|895770|2019| 06| 18|-108.5|        20|        9|
|897340|2019| 07| 19|-108.5|         5|       10|
|895770|2019| 06| 05|-108.1|        20|       11|
|896060|2019| 06| 18|-107.9|         4|       12|
|896060|2019| 06| 06|-107.8|         4|       13|
|897340|2019| 07| 20|-107.6|         7|       14|
|895770|2019| 06| 03|-107.6|        16|       15|
|896060|2019| 05| 21|-107.1|         4|       16|
|895770|2019| 05| 26|-106.8|        19|       17|


In [66]:
temp_per_month_desc = Window.partitionBy("mo").orderBy(
    F.col("count_temp").desc()
)

df_gsod.withColumn(
    "row_number", F.row_number().over(temp_per_month_desc)
).select("stn", "year", "mo", "da", "temp", "count_temp", "row_number").show()

[Stage 99:>                                                         (0 + 1) / 1]

+------+----+---+---+-----+----------+----------+
|   stn|year| mo| da| temp|count_temp|row_number|
+------+----+---+---+-----+----------+----------+
|998012|2018| 01| 27| 44.6|        24|         1|
|994973|2017| 01| 31| 27.1|        24|         2|
|725342|2018| 01| 12| 27.8|        24|         3|
|702625|2017| 01| 18|-44.6|        24|         4|
|994973|2018| 01| 13|  6.7|        24|         5|
|997787|2017| 01| 17| 41.6|        24|         6|
|702220|2018| 01| 24|-41.8|        24|         7|
|994979|2017| 01| 15| 23.5|        24|         8|
|723409|2018| 01| 08| 41.3|        24|         9|
|998282|2017| 01| 08| 33.3|        24|        10|
|994973|2018| 01| 04|  7.9|        24|        11|
|702220|2017| 01| 18|-43.4|        24|        12|
|998012|2018| 01| 25| 31.4|        24|        13|
|997737|2017| 01| 12|  3.1|        24|        14|
|994973|2018| 01| 10| 35.1|        24|        15|
|998012|2017| 01| 08| 18.3|        24|        16|
|997140|2018| 01| 08| 28.3|        24|        17|


## **Analytic functions**

In [68]:
df_gsod.withColumn(
    "previous_temp", F.lag("temp").over(temp_each_year)
).withColumn(
    "previous_temp_2", F.lag("temp", 2).over(temp_each_year)
).select("stn", "year", "mo", "da", "temp", "count_temp", "previous_temp", "previous_temp_2").show()

[Stage 105:>                                                        (0 + 1) / 1]

+------+----+---+---+------+----------+-------------+---------------+
|   stn|year| mo| da|  temp|count_temp|previous_temp|previous_temp_2|
+------+----+---+---+------+----------+-------------+---------------+
|895770|2019| 06| 15|-114.7|        21|         NULL|           NULL|
|895770|2019| 06| 14|-113.6|        20|       -114.7|           NULL|
|896060|2019| 06| 07|-112.4|         4|       -113.6|         -114.7|
|896060|2019| 06| 08|-111.3|         4|       -112.4|         -113.6|
|895770|2019| 06| 17|-110.7|        20|       -111.3|         -112.4|
|895770|2019| 05| 25|-110.3|        20|       -110.7|         -111.3|
|896060|2019| 06| 09|-109.0|         4|       -110.3|         -110.7|
|895770|2019| 06| 02|-108.6|        21|       -109.0|         -110.3|
|895770|2019| 06| 18|-108.5|        20|       -108.6|         -109.0|
|897340|2019| 07| 19|-108.5|         5|       -108.5|         -108.6|
|895770|2019| 06| 05|-108.1|        20|       -108.5|         -108.5|
|896060|2019| 06| 18

In [69]:
df_gsod.withColumn(
    "percent_rank", F.percent_rank().over(temp_each_year)
).withColumn(
    "cume_dist", F.cume_dist().over(temp_each_year)
).select("stn", "year", "mo", "da", "temp", "count_temp", "percent_rank", "cume_dist").show()

[Stage 108:>                                                        (0 + 1) / 1]

+------+----+---+---+------+----------+--------------------+--------------------+
|   stn|year| mo| da|  temp|count_temp|        percent_rank|           cume_dist|
+------+----+---+---+------+----------+--------------------+--------------------+
|895770|2019| 06| 15|-114.7|        21|                 0.0|2.406128505548773E-7|
|895770|2019| 06| 14|-113.6|        20|2.406129084494351E-7|4.812257011097546E-7|
|896060|2019| 06| 07|-112.4|         4|4.812258168988702E-7|7.218385516646319E-7|
|896060|2019| 06| 08|-111.3|         4|7.218387253483052E-7|9.624514022195092E-7|
|895770|2019| 06| 17|-110.7|        20|9.624516337977404E-7|1.203064252774386...|
|895770|2019| 05| 25|-110.3|        20|1.203064542247175...|1.443677103329263...|
|896060|2019| 06| 09|-109.0|         4|1.443677450696610...|1.684289953884141E-6|
|895770|2019| 06| 02|-108.6|        21|1.684290359146045...|1.924902804439018...|
|895770|2019| 06| 18|-108.5|        20|1.924903267595480...|2.406128505548773E-6|
|897340|2019| 07